# Zomato Delivery Operations Analysis Using PySpark
## Stage 2: Data Preprocessing, Data Validation & Feature Engineering

**Author:** Computer Engineering Mini-Project  
**Framework:** Apache Spark via PySpark  
**Dataset:** `Zomato Dataset.csv`  
**Target Parquet Storage:** `data/processed/zomato_cleaned.parquet`

---

### Objectives of Stage 2:
1. **Handle Missing Values:** Reconcile literal string `'NaN'` values with SQL `NULL` and assign explicit `'Unknown'` categories for categorical attributes.
2. **Validate and Sanitize Data:** Correct inverted negative GPS signs, detect Null Island sensor dropouts (`0.0, 0.0`), and cap out-of-bounds ratings (`6.0` -> `5.0`).
3. **Multi-Format Timestamp Parsing:** Standardize `HH:MM`, `HH:MM:SS`, and Microsoft Excel serial day fractions into minute-of-day metrics.
4. **Feature Engineering:**
   - Date attributes: `Order_Month`, `Order_Day`, `Day_of_Week`, `Is_Weekend`
   - Operational timing: `Order_Hour`, `Pickup_Hour`, `Order_to_Pickup_Minutes` (with midnight rollover logic)
   - Geospatial distance: `Approx_Distance_Km` using Spherical Haversine calculation
5. **Before vs. After Telemetry Comparison:** Verify 100% record retention (0 dropped rows) and complete feature readiness.

### Step 1: Initialize PySpark Session

In [1]:
import os
import sys

# Add project root to sys.path
sys.path.append(os.path.abspath('..'))

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

from src.data_loader import get_spark_session, load_raw_dataset
from src.preprocessing import clean_missing_and_types, validate_and_correct_anomalies, engineer_features

spark = get_spark_session('Stage2_Preprocessing_Notebook')
print(f'Spark Master: {spark.sparkContext.master}')
print(f'Spark Version: {spark.version}')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/13 23:03:43 WARN Utils: Your hostname, Rajdeeps-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.35 instead (on interface en0)
26/10/13 23:03:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/13 23:03:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Master: local[*]
Spark Version: 4.2.0


### Step 2: Load the Raw Dataset
We begin by reading the raw 45,584 records.

In [2]:
csv_path = '../data/raw/Zomato Dataset.csv' if os.path.exists('../data/raw/Zomato Dataset.csv') else '../Zomato Dataset.csv'
raw_df = load_raw_dataset(spark, csv_path)
print(f'Raw Record Count: {raw_df.count():,}')
print(f'Raw Column Count: {len(raw_df.columns)}')

Raw Record Count: 45,584
Raw Column Count: 20


### Step 3: Clean Missing Values and Data Types
- Replaces `'NaN'` strings with `'Unknown'` in categorical features (`Weather_conditions`, `Road_traffic_density`, `Festival`, `City`).
- Converts `'NaN'` to `NULL` for numeric features.
- Parses `Order_Date` to Spark `DateType` (`dd-MM-yyyy`).
- Corrects negative coordinates using `abs()`.

In [3]:
cleaned_df = clean_missing_and_types(raw_df)
cleaned_df.printSchema()
cleaned_df.select('ID', 'Weather_conditions', 'Road_traffic_density', 'City', 'Order_Date').show(5)

root
 |-- ID: string (nullable = true)
 |-- Delivery_person_ID: string (nullable = true)
 |-- Delivery_person_Age: string (nullable = true)
 |-- Delivery_person_Ratings: string (nullable = true)
 |-- Restaurant_latitude: double (nullable = true)
 |-- Restaurant_longitude: double (nullable = true)
 |-- Delivery_location_latitude: double (nullable = true)
 |-- Delivery_location_longitude: double (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Time_Orderd: string (nullable = true)
 |-- Time_Order_picked: string (nullable = true)
 |-- Weather_conditions: string (nullable = true)
 |-- Road_traffic_density: string (nullable = true)
 |-- Vehicle_condition: integer (nullable = true)
 |-- Type_of_order: string (nullable = true)
 |-- Type_of_vehicle: string (nullable = true)
 |-- multiple_deliveries: string (nullable = true)
 |-- Festival: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Time_taken (min): integer (nullable = true)
 |-- Delivery_person_Age_Raw: int

### Step 4: Data Validation and Numerical Imputation
- Caps ratings > 5.0 to 5.0.
- Imputes missing `Delivery_person_Age`, `Delivery_person_Ratings`, and `multiple_deliveries` with medians.
- Preserves raw un-imputed values for transparency.

In [4]:
validated_df, medians = validate_and_correct_anomalies(cleaned_df)
print('Medians applied for imputation:', medians)
validated_df.select('Delivery_person_Age', 'Delivery_person_Ratings', 'multiple_deliveries').describe().show()

Medians applied for imputation: {'age_median': 30.0, 'rating_median': 4.7, 'mult_deliveries_median': 1.0}


+-------+-------------------+-----------------------+-------------------+
|summary|Delivery_person_Age|Delivery_person_Ratings|multiple_deliveries|
+-------+-------------------+-----------------------+-------------------+
|  count|              45584|                  45584|              45584|
|   mean|  29.58452527202527|      4.635383029132988| 0.7501974376974377|
| stddev|  5.696220511176436|    0.32485336494182077| 0.5674653654911291|
|    min|                 15|                    1.0|                  0|
|    max|                 50|                    5.0|                  3|
+-------+-------------------+-----------------------+-------------------+



### Step 5: Feature Engineering
We now engineer:
1. `Order_Month`, `Order_Day`, `Day_of_Week`, `Is_Weekend`
2. `Order_Hour`, `Pickup_Hour`, and `Order_to_Pickup_Minutes`
3. `Approx_Distance_Km` (Haversine straight-line distance, masking Null Island coords as `NULL`)

In [5]:
featured_df = engineer_features(validated_df)
print(f'Enriched Dataset Columns: {len(featured_df.columns)}')
featured_df.select(
    'Order_Date', 'Day_of_Week', 'Is_Weekend',
    'Order_Hour', 'Pickup_Hour', 'Order_to_Pickup_Minutes',
    'Approx_Distance_Km', 'Time_taken (min)'
).show(10)

Enriched Dataset Columns: 31


+----------+-----------+----------+----------+-----------+-----------------------+------------------+----------------+
|Order_Date|Day_of_Week|Is_Weekend|Order_Hour|Pickup_Hour|Order_to_Pickup_Minutes|Approx_Distance_Km|Time_taken (min)|
+----------+-----------+----------+----------+-----------+-----------------------+------------------+----------------+
|2022-02-12|        Sat|         1|        21|         22|                     15|             10.28|              46|
|2022-02-13|        Sun|         1|        14|         15|                     10|              6.24|              23|
|2022-03-04|        Fri|         0|        17|         17|                     10|             13.79|              21|
|2022-02-13|        Sun|         1|         9|          9|                     10|              2.93|              20|
|2022-02-14|        Mon|         0|        19|         20|                     15|              19.4|              41|
|2022-04-02|        Sat|         1|        20|  

### Step 6: Before vs. After Preprocessing Comparison
We compare key metrics before and after the cleaning pipeline.

In [6]:
import pandas as pd

comparison_data = [
    ('Total Rows', f'{raw_df.count():,}', f'{featured_df.count():,}'),
    ('Total Columns', len(raw_df.columns), len(featured_df.columns)),
    ('Missing Age', '1,854 (NaN string)', '0 (Imputed with median 30)'),
    ('Missing Ratings', '1,908 (NaN string)', '0 (Imputed with median 4.7)'),
    ('Ratings == 6.0', '53 records', '0 records (Capped to 5.0)'),
    ('Negative Latitudes', '431 records', '0 records (abs() corrected)'),
    ('Null Island (0,0) Coords', '3,640 records', '3,640 records (masked as NULL distance)'),
    ('Approx_Distance_Km created', 'No', f"{featured_df.filter(F.col('Approx_Distance_Km').isNotNull()).count():,} valid distances"),
    ('Order_to_Pickup_Minutes created', 'No', f"{featured_df.filter(F.col('Order_to_Pickup_Minutes').isNotNull()).count():,} valid durations")
]

comparison_df = pd.DataFrame(comparison_data, columns=['Metric / Feature', 'Before Preprocessing', 'After Preprocessing'])
print(comparison_df.to_string(index=False))

               Metric / Feature Before Preprocessing                     After Preprocessing
                     Total Rows               45,584                                  45,584
                  Total Columns                   20                                      31
                    Missing Age   1,854 (NaN string)              0 (Imputed with median 30)
                Missing Ratings   1,908 (NaN string)             0 (Imputed with median 4.7)
                 Ratings == 6.0           53 records               0 records (Capped to 5.0)
             Negative Latitudes          431 records             0 records (abs() corrected)
       Null Island (0,0) Coords        3,640 records 3,640 records (masked as NULL distance)
     Approx_Distance_Km created                   No                  41,944 valid distances
Order_to_Pickup_Minutes created                   No                  43,853 valid durations


### Step 7: Verify Storage in Parquet Format
We verify that the cleaned dataset is successfully persisted in Apache Parquet format for downstream high-performance querying in Stages 3, 4, and 5.

In [7]:
parquet_path = '../data/processed/zomato_cleaned.parquet'
if os.path.exists(parquet_path):
    parquet_df = spark.read.parquet(parquet_path)
    print(f'Successfully loaded Parquet dataset: {parquet_df.count():,} records, {len(parquet_df.columns)} columns.')
else:
    print('Parquet path does not exist yet. Running pipeline export...')
    featured_df.write.mode('overwrite').parquet(parquet_path)
    print('Parquet dataset exported.')

Successfully loaded Parquet dataset: 45,584 records, 31 columns.


### Step 8: Shutdown Spark Session

In [8]:
spark.stop()
print('SparkSession stopped.')

SparkSession stopped.
